# Spread + Mid Prediction

optimal bps with oracle or BTCUSDT: 0.7055
optimal bps with oracle or ETHUSDT: 1.6091
optimal bps with oracle or LTCUSDT: 2.6329
optimal bps with oracle or SOLUSDT: 1.1299



In [1]:
# ================================
# 1. Clone repo
# ================================
!git clone https://github.com/JosephZacharyGawlik/Ultramarin.git
%cd Ultramarin
!git fetch origin
!git checkout -b spread origin/spread


# ================================
# 2. Install dependencies
# ================================
!pip install --upgrade pip
!pip install .


# ================================
# 3. Mount Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')


# ================================
# 4. Set data path
# ================================
DATA_PATH = "/content/drive/MyDrive/Ultramarin Data/data"

import os
print("Files in dataset folder:")
print(os.listdir(DATA_PATH))


# ================================
# 5. GPU check
# ================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

fatal: destination path 'Ultramarin' already exists and is not an empty directory.
/content/Ultramarin
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 8), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 51.99 KiB | 1.49 MiB/s, done.
From https://github.com/JosephZacharyGawlik/Ultramarin
   2eb21e2..94090bb  spread     -> origin/spread
   30cd31e..9371440  potentially-improve-model -> origin/potentially-improve-model
fatal: A branch named 'spread' already exists.
Processing ./.
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'file:///content/Ultramarin'

In [25]:
import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cuda


In [155]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [253]:
import sys, os

# Paths and volume_to_fill
# root_path = DATA_PATH
root_path = "../data"
root = Path(root_path) if Path(root_path).exists() else Path.cwd()

import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_assets = [folder_name for folder_name in os.listdir(root) if "USDT" in folder_name]
data_asset = data_assets[6]

symbol_dir = root / data_asset
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
X_test_path = symbol_dir / "X_test.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("data asset:", data_asset)
print("volume_to_fill:", volume_to_fill)


data asset: XRPUSDT
volume_to_fill: 13736.0


In [254]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS + ["target"]

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)} (20 LOB + mid_price)")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 21 (20 LOB + mid_price)


In [255]:
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# Model Training

In [257]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 100, 
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 600,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    x_test_path = X_test_path,
    feature_cols = FEATURE_COLS,
)

model, train_scalers, val_loader, val_id_map, processor = train_val(config)

Epoch 01 | Train: 0.662864 | Val: 1.020876
Epoch 02 | Train: 0.318760 | Val: 2.101875
Epoch 03 | Train: 0.138982 | Val: 0.839144
Epoch 04 | Train: 0.100468 | Val: 0.459085
Epoch 05 | Train: 0.083812 | Val: 0.418555
Epoch 06 | Train: 0.084466 | Val: 0.661689
Epoch 07 | Train: 0.083224 | Val: 0.392256
Epoch 08 | Train: 0.078014 | Val: 0.536330
Epoch 09 | Train: 0.076591 | Val: 0.416442
Epoch 10 | Train: 0.075761 | Val: 0.334872
Epoch 11 | Train: 0.072988 | Val: 0.325483
Epoch 12 | Train: 0.072676 | Val: 0.371736
Epoch 13 | Train: 0.073329 | Val: 0.351170
Epoch 14 | Train: 0.071711 | Val: 0.330042
Epoch 15 | Train: 0.071431 | Val: 0.350339
Epoch 16 | Train: 0.070996 | Val: 0.348044
Epoch 17 | Train: 0.070163 | Val: 0.366177
Epoch 18 | Train: 0.069937 | Val: 0.458267
Epoch 19 | Train: 0.069769 | Val: 0.311183
Epoch 20 | Train: 0.068792 | Val: 0.338546
Epoch 21 | Train: 0.068533 | Val: 0.302622
Epoch 22 | Train: 0.068836 | Val: 0.466354
Epoch 23 | Train: 0.070145 | Val: 0.359313
Epoch 24 | 

# Mid Shift Model and BPS

In [258]:
config.target = "mid shift"

model_shift, train_scalers_shift, val_loader_shift, val_id_map_shift, processor_shift = train_val(config)

Epoch 01 | Train: 0.636026 | Val: 1.111278
Epoch 02 | Train: 0.260474 | Val: 0.595040
Epoch 03 | Train: 0.119900 | Val: 2.370303
Epoch 04 | Train: 0.090749 | Val: 0.850688
Epoch 05 | Train: 0.082025 | Val: 0.431951
Epoch 06 | Train: 0.078852 | Val: 0.401137
Epoch 07 | Train: 0.079263 | Val: 0.491353
Epoch 08 | Train: 0.078237 | Val: 0.322908
Epoch 09 | Train: 0.074058 | Val: 0.371820
Epoch 10 | Train: 0.073179 | Val: 0.401511
Epoch 11 | Train: 0.073376 | Val: 0.302919
Epoch 12 | Train: 0.071894 | Val: 0.345314
Epoch 13 | Train: 0.071380 | Val: 0.357370
Epoch 14 | Train: 0.070698 | Val: 0.498430
Epoch 15 | Train: 0.070131 | Val: 0.310679
Epoch 16 | Train: 0.071188 | Val: 0.407838
Epoch 17 | Train: 0.069651 | Val: 0.279861
Epoch 18 | Train: 0.069658 | Val: 0.317352
Epoch 19 | Train: 0.068884 | Val: 0.286173
Epoch 20 | Train: 0.068432 | Val: 0.309327
Epoch 21 | Train: 0.067440 | Val: 0.251118
Epoch 22 | Train: 0.066949 | Val: 0.317039
Epoch 23 | Train: 0.067590 | Val: 0.272788
Epoch 24 | 

## Cell

In [259]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model_shift.eval()
all_preds = []

last_mid_train = train_scalers_shift["last_mid_train"]
last_mid_val = train_scalers_shift["last_mid_val"]

mid_shift_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]

val_mid_shift_mean = train_scalers_shift["val_feat_means"][:, :, mid_shift_idx]
val_mid_shift_std  = train_scalers_shift["val_feat_stds"][:, :, mid_shift_idx]

with torch.no_grad():
    for x_batch, y_batch, target in val_loader_shift:
        x_batch = x_batch.to(config.device)
        pred = model_shift(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # [num_val_ids, 60]

# denormalize mid shifts
mid_shift_m = val_mid_shift_mean.squeeze(0).cpu().numpy()[:, None]
mid_shift_s = val_mid_shift_std.squeeze(0).cpu().numpy()[:, None]

val_preds = val_preds * mid_shift_s + mid_shift_m

# convert mid shift -> predicted mid
val_preds = val_preds + last_mid_val.values[:, None]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map_shift.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_shift_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_shift_bps.append(impl_error * vol_penalty)

model_shift_bps = np.array(model_shift_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid shift")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_shift_bps)}")
print(f"Mean:   {model_shift_bps.mean():.4f} bps")
print(f"Median: {np.median(model_shift_bps):.4f} bps")
print(f"Std:    {model_shift_bps.std():.4f} bps")
print(f"Min:    {model_shift_bps.min():.4f} bps")
print(f"Max:    {model_shift_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid shift
Instruments evaluated: 127
Mean:   4.0170 bps
Median: 3.4337 bps
Std:    2.9181 bps
Min:    0.0661 bps
Max:    19.5360 bps


In [260]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model_shift.eval()
all_preds = []

last_mid_train = train_scalers_shift["last_mid_train"]
last_mid_val = train_scalers_shift["last_mid_val"]

mid_shift_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]

val_mid_shift_mean = train_scalers_shift["val_feat_means"][:, :, mid_shift_idx]
val_mid_shift_std  = train_scalers_shift["val_feat_stds"][:, :, mid_shift_idx]

with torch.no_grad():
    for x_batch, y_batch, target in val_loader_shift:
        x_batch = x_batch.to(config.device)
        pred = model_shift(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # [num_val_ids, 60]

# denormalize mid shifts
mid_shift_m = val_mid_shift_mean.squeeze(0).cpu().numpy()[:, None]
mid_shift_s = val_mid_shift_std.squeeze(0).cpu().numpy()[:, None]

val_preds = val_preds * mid_shift_s + mid_shift_m

# convert mid shift -> predicted mid
val_preds = val_preds + last_mid_val.values[:, None]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map_shift.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_shift_bps_weighting = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()

    # weights for mid are

    quad = np.linspace(1, 60, 60)**2

    weights = weights * quad
    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_shift_bps_weighting.append(impl_error * vol_penalty)

model_shift_bps_weighting = np.array(model_shift_bps_weighting)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid shift weighting")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_shift_bps_weighting)}")
print(f"Mean:   {model_shift_bps_weighting.mean():.4f} bps")
print(f"Median: {np.median(model_shift_bps_weighting):.4f} bps")
print(f"Std:    {model_shift_bps_weighting.std():.4f} bps")
print(f"Min:    {model_shift_bps_weighting.min():.4f} bps")
print(f"Max:    {model_shift_bps_weighting.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid shift weighting
Instruments evaluated: 127
Mean:   3.6067 bps
Median: 3.4342 bps
Std:    2.3962 bps
Min:    0.0327 bps
Max:    17.7441 bps


# Model-Based Predictions (Spread)

In [261]:
times = pd.read_parquet(Y_path)["time_in_hour"].sort_values(ascending=True).unique()

In [262]:
# train a model for spread
config.target = "spread" 
config.loss = "huber"

model_s, train_scalers_s, val_loader_s, val_id_map_s, processor_s = train_val(config)

Epoch 01 | Train: 0.427502 | Val: 0.449684
Epoch 02 | Train: 0.417437 | Val: 0.444070
Epoch 03 | Train: 0.415024 | Val: 0.449265
Epoch 04 | Train: 0.414226 | Val: 0.449591
Epoch 05 | Train: 0.412172 | Val: 0.450130
Epoch 06 | Train: 0.411207 | Val: 0.450661
Epoch 07 | Train: 0.410892 | Val: 0.441466
Epoch 08 | Train: 0.410510 | Val: 0.441587
Epoch 09 | Train: 0.409937 | Val: 0.447210
Epoch 10 | Train: 0.408936 | Val: 0.448733
Epoch 11 | Train: 0.407152 | Val: 0.442581
Epoch 12 | Train: 0.407865 | Val: 0.455906
Epoch 13 | Train: 0.408494 | Val: 0.447736
Epoch 14 | Train: 0.405762 | Val: 0.443798
Epoch 15 | Train: 0.404562 | Val: 0.445467
Epoch 16 | Train: 0.402460 | Val: 0.453680
Epoch 17 | Train: 0.402251 | Val: 0.448663
Epoch 18 | Train: 0.402260 | Val: 0.450990
Epoch 19 | Train: 0.401876 | Val: 0.445133
Epoch 20 | Train: 0.401333 | Val: 0.446664
Epoch 21 | Train: 0.402802 | Val: 0.451644
Epoch 22 | Train: 0.399983 | Val: 0.450175
Epoch 23 | Train: 0.399527 | Val: 0.450279
Epoch 24 | 

## Cell

In [263]:
# Implementation error for the spread model
 
# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_s_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    spread_preds = val_preds[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights = norm_w / norm_w.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_s_bps.append(impl_error * vol_penalty)

model_s_bps = np.array(model_s_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR spread")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_s_bps)}")
print(f"Mean:   {model_s_bps.mean():.4f} bps")
print(f"Median: {np.median(model_s_bps):.4f} bps")
print(f"Std:    {model_s_bps.std():.4f} bps")
print(f"Min:    {model_s_bps.min():.4f} bps")
print(f"Max:    {model_s_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR spread
Instruments evaluated: 127
Mean:   4.1698 bps
Median: 3.6748 bps
Std:    3.0534 bps
Min:    0.1351 bps
Max:    19.6593 bps


In [264]:
# Implementation error for the spread model
 
# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_s_bps_weighting = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    spread_preds = val_preds[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights = norm_w / norm_w.sum()

    # weights for spread are linear K=30 spread

    lin = np.zeros_like(weights)
    lin[-30:] = np.linspace(1, 30, 30)

    weights = weights * lin
    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_s_bps_weighting.append(impl_error * vol_penalty)

model_s_bps_weighting = np.array(model_s_bps_weighting)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR spread weighting")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_s_bps_weighting)}")
print(f"Mean:   {model_s_bps_weighting.mean():.4f} bps")
print(f"Median: {np.median(model_s_bps_weighting):.4f} bps")
print(f"Std:    {model_s_bps_weighting.std():.4f} bps")
print(f"Min:    {model_s_bps_weighting.min():.4f} bps")
print(f"Max:    {model_s_bps_weighting.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR spread weighting
Instruments evaluated: 127
Mean:   3.6852 bps
Median: 3.4725 bps
Std:    2.4936 bps
Min:    0.0041 bps
Max:    16.6420 bps


## Positions

In [11]:
# --- 1. Generate predictions using test_loader ---
model.eval()
test_preds = []

test_loader, scalers, test_map, processor = generate_test_loader(config, processor)

with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)  # Shape: [num_test_ids, 60]

# Get test IDs
test_ids = [int(uid) for idx, uid in test_map.cpu().numpy()]
print(f"Test predictions shape: {test_preds.shape}")
print(f"Number of testing instruments: {len(test_ids)}")

test_positions = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Model-based positions
    spread_preds = test_preds[i]

    # spread weights
    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights = norm_w / norm_w.sum()
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions = pd.concat([test_positions, df], ignore_index=True)

test_positions

Test predictions shape: (303, 60)
Number of testing instruments: 303


,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.004816
2,5.692284e+16,0 days 00:59:02,0.006147
3,5.692284e+16,0 days 00:59:03,0.008299
4,5.692284e+16,0 days 00:59:04,0.010694
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.005958
18176,1.841373e+19,0 days 00:59:56,0.004416
18177,1.841373e+19,0 days 00:59:57,0.002910
18178,1.841373e+19,0 days 00:59:58,0.001438


# Model-Based Predictions (Mid)

## Cell

In [265]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps.append(impl_error * vol_penalty)

model_bps = np.array(model_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_bps)}")
print(f"Mean:   {model_bps.mean():.4f} bps")
print(f"Median: {np.median(model_bps):.4f} bps")
print(f"Std:    {model_bps.std():.4f} bps")
print(f"Min:    {model_bps.min():.4f} bps")
print(f"Max:    {model_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid
Instruments evaluated: 127
Mean:   4.0170 bps
Median: 3.4383 bps
Std:    2.9149 bps
Min:    0.0627 bps
Max:    19.5380 bps


In [266]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps_weighting = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()

    # weights for mid are quadratic K=60

    quad = np.linspace(1, 60, 60)**2

    weights = weights * quad
    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps_weighting.append(impl_error * vol_penalty)

model_bps_weighting = np.array(model_bps_weighting)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid weighting")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_bps_weighting)}")
print(f"Mean:   {model_bps_weighting.mean():.4f} bps")
print(f"Median: {np.median(model_bps_weighting):.4f} bps")
print(f"Std:    {model_bps_weighting.std():.4f} bps")
print(f"Min:    {model_bps_weighting.min():.4f} bps")
print(f"Max:    {model_bps_weighting.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid weighting
Instruments evaluated: 127
Mean:   3.6066 bps
Median: 3.4341 bps
Std:    2.3963 bps
Min:    0.0387 bps
Max:    17.7450 bps


## Positions

In [13]:
# --- 1. Generate predictions using test_loader ---
model.eval()
test_preds = []

test_loader, scalers, test_map, processor = generate_test_loader(config, processor)

with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)  # Shape: [num_test_ids, 60]

# Get test IDs
test_ids = [int(uid) for idx, uid in test_map.cpu().numpy()]
print(f"Test predictions shape: {test_preds.shape}")
print(f"Number of testing instruments: {len(test_ids)}")

test_positions = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Model-based positions
    mid_preds = test_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - (price_distance / price_distance.sum())

    weights /= weights.sum()
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions = pd.concat([test_positions, df], ignore_index=True)

test_positions

Test predictions shape: (303, 60)
Number of testing instruments: 303


,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.065446
1,5.692284e+16,0 days 00:59:01,0.065510
2,5.692284e+16,0 days 00:59:02,0.065528
3,5.692284e+16,0 days 00:59:03,0.065557
4,5.692284e+16,0 days 00:59:04,0.065590
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.067677
18176,1.841373e+19,0 days 00:59:56,0.067708
18177,1.841373e+19,0 days 00:59:57,0.067738
18178,1.841373e+19,0 days 00:59:58,0.067768


# Model-Based Predictions (Mid + Spread)

The ideal model that combines Mid and Spread as evaluated previously was 1/6 quadratic K=60 mid + 5/6 linear K=30 spread.

In [267]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds_s = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds_s.append(pred.cpu().numpy())

val_preds_s = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_ms_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()

    ### spread
    # Model-based positions
    spread_preds = val_preds_s[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights_s = norm_w / norm_w.sum()
    ###

    weights_final = (1/6) * weights + (5/6) * weights_s

    # normalize
    weights_final /= weights_final.sum()
    
    model_positions = weights_final * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_ms_bps.append(impl_error * vol_penalty)

model_ms_bps = np.array(model_ms_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid + spread")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_ms_bps)}")
print(f"Mean:   {model_ms_bps.mean():.4f} bps")
print(f"Median: {np.median(model_ms_bps):.4f} bps")
print(f"Std:    {model_ms_bps.std():.4f} bps")
print(f"Min:    {model_ms_bps.min():.4f} bps")
print(f"Max:    {model_ms_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid + spread
Instruments evaluated: 127
Mean:   4.1474 bps
Median: 3.7570 bps
Std:    3.4141 bps
Min:    0.0334 bps
Max:    24.7499 bps


In [268]:
# Implementation error for the model
 
# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# --- 1. Generate predictions using val_loader ---
model_s.eval()
all_preds_s = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None)
        all_preds_s.append(pred.cpu().numpy())

val_preds_s = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_ms_bps_weighting = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_distance = abs(pred_close - mid_preds)
    weights = 1 - price_distance / price_distance.sum()

    weights /= weights.sum()

    ### spread
    # Model-based positions
    spread_preds = val_preds_s[i]

    # spread weights

    raw_w = 1 / spread_preds

    w_min = raw_w.min()
    w_max = raw_w.max()

    norm_w = (raw_w - w_min) / (w_max - w_min)

    weights_s = norm_w / norm_w.sum()
    ###

    # --- quadratic K=60 ---
    quad = np.linspace(1, 60, 60)**2
    quad /= quad.sum()

    # --- linear K=30 (only last 30 count) ---
    lin = np.zeros(60)
    lin_part = np.linspace(1, 30, 30)
    lin_part /= lin_part.sum()
    lin[-30:] = lin_part

    # --- combine ---
    w_quad = quad * weights
    w_lin = lin * weights_s

    weights_final = (1/6) * w_quad + (5/6) * w_lin

    # normalize
    weights_final /= weights_final.sum()
    
    model_positions = weights_final * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_ms_bps_weighting.append(impl_error * vol_penalty)

model_ms_bps_weighting = np.array(model_ms_bps_weighting)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR mid + spread weighting")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_ms_bps_weighting)}")
print(f"Mean:   {model_ms_bps_weighting.mean():.4f} bps")
print(f"Median: {np.median(model_ms_bps_weighting):.4f} bps")
print(f"Std:    {model_ms_bps_weighting.std():.4f} bps")
print(f"Min:    {model_ms_bps_weighting.min():.4f} bps")
print(f"Max:    {model_ms_bps_weighting.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR mid + spread weighting
Instruments evaluated: 127
Mean:   3.5920 bps
Median: 3.4486 bps
Std:    2.4915 bps
Min:    0.0192 bps
Max:    19.5085 bps


# TWAP

## Cell

In [269]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 60  # TWAP window: last K seconds

baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"Min:    {baseline_bps.min():.4f} bps")
print(f"Max:    {baseline_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-60s) IMPLEMENTATION ERROR
Instruments evaluated: 127
Mean:   4.0240 bps
Median: 3.4486 bps
Std:    2.9275 bps
Min:    0.0775 bps
Max:    19.5510 bps


In [270]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 20  # TWAP window: last K seconds

baseline_14_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_14_bps.append(impl_error * vol_penalty)

baseline_14_bps = np.array(baseline_14_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_14_bps)}")
print(f"Mean:   {baseline_14_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_14_bps):.4f} bps")
print(f"Std:    {baseline_14_bps.std():.4f} bps")
print(f"Min:    {baseline_14_bps.min():.4f} bps")
print(f"Max:    {baseline_14_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-20s) IMPLEMENTATION ERROR
Instruments evaluated: 127
Mean:   3.5670 bps
Median: 3.3628 bps
Std:    2.4593 bps
Min:    0.0333 bps
Max:    17.2992 bps


## Positions

In [60]:
test_positions_twap = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    weights = 1 / 60
    
    model_positions = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions_twap = pd.concat([test_positions_twap, df], ignore_index=True)

test_positions_twap

NameError: name 'test_ids' is not defined

# Predictive_Scheduler Predictions

## Cell

In [271]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=K_SECONDS,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

scheduler_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Use predictive_scheduler to build positions
    mid_preds = val_preds[i]
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )
    
    # Simulate
    sched_vol, sched_avg_price = simulate_walk_the_book(
        scheduler_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if sched_vol > 0 and not np.isnan(sched_avg_price):
        impl_error = np.abs(sched_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / sched_vol)
        scheduler_bps.append(impl_error * vol_penalty)

scheduler_bps = np.array(scheduler_bps)

print(f"\n{'='*50}")
print(f"PREDICTIVE SCHEDULER IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Config: window={sched_cfg.window}, alpha={sched_cfg.alpha}")
print(f"Instruments evaluated: {len(scheduler_bps)}")
print(f"Mean:   {scheduler_bps.mean():.4f} bps")
print(f"Median: {np.median(scheduler_bps):.4f} bps")
print(f"Std:    {scheduler_bps.std():.4f} bps")
print(f"Min:    {scheduler_bps.min():.4f} bps")
print(f"Max:    {scheduler_bps.max():.4f} bps")
print(f"{'='*50}")


PREDICTIVE SCHEDULER IMPLEMENTATION ERROR
Config: window=20, alpha=0.1
Instruments evaluated: 127
Mean:   3.5267 bps
Median: 3.3805 bps
Std:    2.3098 bps
Min:    0.0718 bps
Max:    15.4946 bps


## Positions

In [ ]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=14,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

test_positions_scheduler = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Use predictive_scheduler to build positions
    mid_preds = test_preds[i]
    
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': scheduler_positions
    })

    test_positions_scheduler = pd.concat([test_positions_scheduler, df], ignore_index=True)

test_positions_scheduler

,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.000000
2,5.692284e+16,0 days 00:59:02,0.000000
3,5.692284e+16,0 days 00:59:03,0.000000
4,5.692284e+16,0 days 00:59:04,0.000000
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.286168
18176,1.841373e+19,0 days 00:59:56,0.286168
18177,1.841373e+19,0 days 00:59:57,0.286168
18178,1.841373e+19,0 days 00:59:58,0.286168


# Save Positions

In [ ]:
len(test_positions["anonymized_id"].unique())

303

In [ ]:
base_path = Path(f"positions_joe/{data_asset}")
base_path.mkdir(parents=True, exist_ok=True)

# 2. Map the filenames to the actual DataFrame objects
save_map = {
    "midprice.parquet": test_positions,
    "twap.parquet": test_positions_twap,
    "scheduler.parquet": test_positions_scheduler
}

# 3. Save each with a quick confirmation
for filename, df in save_map.items():
    if df is not None and not df.empty:
        target_path = base_path / filename
        df.to_parquet(target_path, index=False, engine='pyarrow')
        print(f"✅ Saved {len(df)} rows to: {target_path}")
    else:
        print(f"⚠️ Skipping {filename}: DataFrame is empty or None.")

✅ Saved 18180 rows to: positions_joe/BTCUSDT/midprice.parquet
✅ Saved 18180 rows to: positions_joe/BTCUSDT/twap.parquet
✅ Saved 18180 rows to: positions_joe/BTCUSDT/scheduler.parquet


# Evaluation Metrics

We want a file of this format:

MSE, R2, BPS_MID, BPS_TWAP, BPS_SCHEDULER 

In [272]:
model.eval()

# Per-instrument normalization stats for VALIDATION set
mid_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_mid_mean = train_scalers["val_feat_means"][:, :, mid_idx]  # [1, num_val_ids]
val_mid_std  = train_scalers["val_feat_stds"][:, :, mid_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
mid_m = val_mid_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
mid_s = val_mid_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * mid_s + mid_m
preds_denorm = all_preds * mid_s + mid_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

global_mean_raw = targets_denorm.mean().item() 

ss_res = ((targets_denorm - preds_denorm) ** 2).sum().item()
ss_tot = ((targets_denorm - global_mean_raw) ** 2).sum().item()

final_r2 = 1 - (ss_res / ss_tot)

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2_avg = r2_per_inst.mean().item()

final_mse = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE mid: {final_mse:.6f}")
print(f"Denormalized R2 mid:  {final_r2:.6f}")
print(f"Denormalized R2 mid avg:  {final_r2_avg:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # shift

model_shift.eval()

# Per-instrument normalization stats for VALIDATION set
mid_shift_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_mid_shift_mean = train_scalers_shift["val_feat_means"][:, :, mid_shift_idx]  # [1, num_val_ids]
val_mid_shift_std  = train_scalers_shift["val_feat_stds"][:, :, mid_shift_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader_shift:
        x_batch = x_batch.to(config.device)
        pred = model_shift(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
mid_shift_m = val_mid_shift_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
mid_shift_s = val_mid_shift_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * mid_shift_s + mid_shift_m
preds_denorm = all_preds * mid_shift_s + mid_shift_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

global_mean_raw = targets_denorm.mean().item() 

ss_res = ((targets_denorm - preds_denorm) ** 2).sum().item()
ss_tot = ((targets_denorm - global_mean_raw) ** 2).sum().item()

final_r2_shift = 1 - (ss_res / ss_tot)

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2_shift_avg = r2_per_inst.mean().item()

final_mse_shift = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE mid: {final_mse_shift:.6f}")
print(f"Denormalized R2 mid:  {final_r2_shift:.6f}")
print(f"Denormalized R2 mid avg:  {final_r2_shift_avg:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # spread
model_s.eval()

# Per-instrument normalization stats for VALIDATION set
spread_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["target"]
val_spread_mean = train_scalers_s["val_feat_means"][:, :, spread_idx]  # [1, num_val_ids]
val_spread_std  = train_scalers_s["val_feat_stds"][:, :, spread_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model_s(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
# val_mid_mean/val_mid_std shape: [1, num_val_ids] -> [num_val_ids, 1] for broadcasting
spread_m = val_spread_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
spread_s = val_spread_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * spread_s + spread_m
preds_denorm = all_preds * spread_s + spread_m

# --- Step 3: Compute R2 and MSE in raw dollar space ---

global_mean_raw = targets_denorm.mean().item() 

ss_res = ((targets_denorm - preds_denorm) ** 2).sum().item()
ss_tot = ((targets_denorm - global_mean_raw) ** 2).sum().item()

final_r2_s = 1 - (ss_res / ss_tot)

# per-instrument means
y_mean = targets_denorm.mean(dim=1, keepdim=True)

# sums of squares per instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)

r2_per_inst = 1 - ss_res / ss_tot
final_r2_s_avg = r2_per_inst.mean().item()

final_mse_s = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE spread: {final_mse_s:.6f}")
print(f"Denormalized R2 spread:  {final_r2_s:.6f}")
print(f"Denormalized R2 spread avg:  {final_r2_s_avg:.6f}")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

metrics_df = pd.DataFrame({
    'data_asset': [data_asset],
    'mse mid': [final_mse],
    'r2 mid': [final_r2],
    'r2 mid avg': [final_r2_avg],
    'mse mid shift': [final_mse_shift],
    'r2 mid shift': [final_r2_shift],
    'r2 mid shift avg': [final_r2_shift_avg],
    'mse spread': [final_mse_s],
    'r2 spread': [final_r2_s],
    'r2 spread avg': [final_r2_s_avg],
    'bps_mid': [model_bps.mean().item()],
    'bps_mid_weighting': [model_bps_weighting.mean().item()],
    'bps_mid_shift': [model_shift_bps.mean().item()],
    'bps_mid_shift_weighting': [model_shift_bps_weighting.mean().item()],
    'bps_spread': [model_s_bps.mean().item()],
    'bps_spread_weighting': [model_s_bps_weighting.mean().item()],
    'bps_mid_spread': [model_ms_bps.mean().item()],
    'bps_mid_spread_weighting': [model_ms_bps_weighting.mean().item()],
    'bps_twap': [baseline_bps.mean().item()],
    'bps_14_twap': [baseline_14_bps.mean().item()],
    'bps_scheduler': [scheduler_bps.mean().item()]
})

target_path = f"eval_df_{data_asset}.parquet"
metrics_df.to_parquet(target_path, index=False, engine='pyarrow')
print(f"Saved eval metrics to: {target_path}")
metrics_df

Denormalized MSE mid: 0.000011
Denormalized R2 mid:  0.999978
Denormalized R2 mid avg:  -13.620461
Denormalized MSE mid: 0.000004
Denormalized R2 mid:  0.201424
Denormalized R2 mid avg:  -4.095814
Denormalized MSE spread: 0.000004
Denormalized R2 spread:  0.098759
Denormalized R2 spread avg:  -123.779144
Saved eval metrics to: eval_df_XRPUSDT.parquet


,data_asset,mse mid,r2 mid,r2 mid avg,mse mid shift,r2 mid shift,r2 mid shift avg,mse spread,r2 spread,r2 spread avg,bps_mid,bps_mid_weighting,bps_mid_shift,bps_mid_shift_weighting,bps_spread,bps_spread_weighting,bps_mid_spread,bps_mid_spread_weighting,bps_twap,bps_14_twap,bps_scheduler
0,XRPUSDT,0.000011,0.999978,-13.620461,0.000004,0.201424,-4.095814,0.000004,0.098759,-123.779144,4.017015,3.606639,4.017015,3.606725,4.169797,3.685176,4.14742,3.592005,4.02398,3.567023,3.526738


# Test Predictions

In [ ]:
from utils.test import generate_test_predictions

# --- Execution ---
test_preds, test_id_map = generate_test_predictions(model, config, processor, num_ids=5)

* **The Map:** This `test_id_np` array pairs the model's row index with the actual anonymized ID.

In [ ]:
# Convert test_id_map to numpy (handles both CPU and GPU tensors)
if hasattr(test_id_map, 'cpu'):
    test_id_np = test_id_map.cpu().numpy().astype(np.uint64)
else:
    test_id_np = np.array(test_id_map, dtype=np.uint64)

# Create a quick summary for inspection
print(f"{'Anonymized ID':<25} | {'First Pred (t+1)':<18} | {'Last Pred (t+60)':<18}")
print("-" * 70)

for i in range(len(test_id_np)):  # Look at first 5
    orig_id = test_id_np[i, 1]
    first_p = test_preds[i, 0]
    last_p  = test_preds[i, -1]
    print(f"{str(orig_id):<25} | {first_p:18.6f} | {last_p:18.6f}")

In [ ]:
# 1. Load your data into Polars
# Using test_id_np[:, 1] for our big IDs
id_df = pl.DataFrame({
    "anonymized_id": test_id_np[:, 1],
    "preds": test_preds.tolist()  # This creates a column of lists, each 60 items long
})

# 2. Explode and add the time duration
final_df = (
    id_df.explode("preds")  # This turns each list of 60 into 60 separate rows
    .with_columns(
        # This counts 0 to 59 for every ID
        pl.col("anonymized_id").cum_count().over("anonymized_id").sub(1).alias("step")
    )
    .with_columns(
        # Create the duration starting at 59 minutes
        pl.duration(minutes=59, seconds=pl.col("step")).alias("time_in_hour")
    )
    .select([
        "anonymized_id",
        "time_in_hour",
        pl.col("preds").alias("mid_price_pred")
    ])
)

display(final_df.head(65))

# Final Comparison of Position Prediction

In [ ]:
test_preds.shape

In [ ]:
bla = pd.read_parquet(X_test_path).sort_values(["anonymized_id", "time_in_hour"])
bla["anonymized_id"].unique().shape

In [ ]:
final_df.shape[0] / 60

In [ ]:
final_df["position"].isclose(test_preds[""])